# TripMate: чат-агент на LangChain с памятью фактов в файле

**ДЗ 2.** Агент проводит короткую анкету путешественника и запоминает **факты** о пользователе
(имя, город вылета, направление, даты, бюджет, с кем едет, интересы, особые пожелания).
Память — это **не история чата**, а отдельный профиль в `memory.json`. При следующем запуске агент
читает профиль, не переспрашивает уже известное и продолжает анкету с первого пустого поля.

Как устроено:

| часть | реализация |
|---|---|
| чат-цепочка | `ChatPromptTemplate \| ChatOpenAI \| StrOutputParser`, обёрнутая в `RunnableWithMessageHistory` (история — только в рамках текущего запуска) |
| долговременная память | `FactMemory(BaseChatMemory)`: `load_memory_variables()` читает профиль, `save_context()` извлекает факты и пишет `memory.json` |
| извлечение фактов | отдельный LLM-вызов со structured output (`with_structured_output(TravelerFacts, method="json_schema")`) |
| профили | ключ `user_id` = `session_id`: в одном файле хранится несколько пользователей |
| служебный вывод | отдельная CELL «Служебная панель», которая обновляется на месте через `display_id`; в основной чат служебное не попадает |

Модель — локальная **Qwen2.5-3B-Instruct** через llama.cpp (`scripts/start_local.sh`, порт 8083).

**Как запускать вручную:** раздел 1 (код) → раздел 2 (первая сессия) → *Kernel → Restart* →
снова раздел 1 → раздел 3 (после перезапуска). Ноутбук в репозитории выполнен скриптом
`hw2/run_demo.py`, который делает ровно это: две сессии в двух **разных процессах** ядра (см. PID).

## 1. Код агента

In [1]:
import json
import os
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Callable

from IPython.display import HTML, display
from langchain_classic.memory.chat_memory import BaseChatMemory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, ConfigDict, Field

import warnings
from langchain_core._api import LangChainDeprecationWarning

# Задание рекомендует RunnableWithMessageHistory и BaseChatMemory; в LangChain 1.x они помечены
# устаревшими в пользу LangGraph, но работают. Прячем только это предупреждение.
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

LLM_URL = os.getenv("HW2_LLM_URL", "http://127.0.0.1:8083/v1")
LLM_MODEL = os.getenv("HW2_LLM_MODEL", "qwen2.5-3b-instruct")
MEMORY_PATH = Path("memory.json")

chat_llm = ChatOpenAI(base_url=LLM_URL, api_key="local", model=LLM_MODEL, temperature=0.3)
extract_llm = ChatOpenAI(base_url=LLM_URL, api_key="local", model=LLM_MODEL, temperature=0)

### Схема фактов и анкета

Pydantic-модель одновременно задаёт JSON-схему для structured output и описывает,
какие факты храним. Порядок `PROFILE_QUESTIONS` — порядок вопросов анкеты.

In [2]:
class TravelerFacts(BaseModel):
    """Факты о путешественнике из ОДНОЙ реплики. null — если в реплике этого нет."""
    name: str | None = Field(None, description="Имя пользователя")
    home_city: str | None = Field(None, description="Город, где живёт пользователь и откуда вылетает")
    destination: str | None = Field(None, description="Куда хочет поехать (город или страна)")
    travel_dates: str | None = Field(None, description="Когда и на сколько дней поездка, например: середина ноября, 5 дней")
    budget: str | None = Field(None, description="Бюджет поездки с валютой")
    companions: str | None = Field(None, description="С кем едет: один, с партнёром, с детьми, с друзьями")
    interests: str | None = Field(None, description="Что любит в путешествиях")
    other_facts: list[str] = Field(default_factory=list,
                                   description="Другие важные факты, не подходящие к полям выше: "
                                               "аллергии, страхи, ограничения, особые предпочтения")


class SlotAnswer(BaseModel):
    """Ответ пользователя на ОДИН вопрос анкеты."""
    answered: bool = Field(description="true — пользователь ответил на вопрос по существу; false — уклонился, "
                                       "сказал «не знаю»/«пока не решил» или задал встречный вопрос")
    value: str | None = Field(None, description="Ответ на вопрос кратко, словами пользователя")
    other_facts: list[str] = Field(default_factory=list,
                                   description="Другие важные факты о пользователе из реплики, не относящиеся к вопросу: "
                                               "аллергии, страхи, ограничения, особые предпочтения")


PROFILE_QUESTIONS = {
    "name": "Как вас зовут?",
    "home_city": "Из какого города вы обычно вылетаете?",
    "destination": "Куда хотите поехать?",
    "travel_dates": "Когда планируете поездку и на сколько дней?",
    "budget": "Какой бюджет на поездку?",
    "companions": "С кем поедете — один, с партнёром, с детьми, с друзьями?",
    "interests": "Что вы больше всего любите в путешествиях: музеи, пляж, еду, природу, прогулки?",
}
FACT_LABELS = {
    "name": "Имя", "home_city": "Город вылета", "destination": "Куда хочет поехать",
    "travel_dates": "Планируемые даты поездки", "budget": "Бюджет", "companions": "С кем поедет",
    "interests": "Интересы в путешествиях",
}

`SlotAnswer` — схема для режима анкеты: извлекаем ответ только на заданный вопрос.
Опыт прогонов показал, что 3B-модель, которой дали сразу все 7 полей, путает их
(«аллергия на орехи» уезжала в интересы), а из встречного вопроса «а в ноябре там не холодно?»
выдумывала даты. `TravelerFacts` используется после анкеты, когда пользователь может поменять любой факт.

### Хранилище: `memory.json`

Формат: `{"users": {user_id: {"facts": {...}, "other_facts": [...], "updated_at": ...}}}`.
Нет файла — пустая память. Запись атомарная (через временный файл), чтобы падение
посреди записи не испортило профиль.

In [3]:
class FactStore:
    def __init__(self, path: Path):
        self.path = Path(path)

    def _read(self) -> dict:
        if not self.path.exists():
            return {"users": {}}
        return json.loads(self.path.read_text(encoding="utf-8"))

    def _write(self, data: dict) -> None:
        tmp = self.path.with_suffix(".tmp")
        tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        tmp.replace(self.path)

    def load(self, user_id: str) -> dict:
        profile = self._read()["users"].get(user_id, {})
        return {"facts": profile.get("facts", {}), "other_facts": profile.get("other_facts", [])}

    def save(self, user_id: str, profile: dict) -> None:
        data = self._read()
        data["users"][user_id] = {**profile, "updated_at": datetime.now().isoformat(timespec="seconds")}
        self._write(data)

    def delete(self, user_id: str) -> None:
        data = self._read()
        if data["users"].pop(user_id, None) is not None:
            self._write(data)

### `FactMemory` — своя память-наследник `BaseChatMemory`

* `load_memory_variables()` отдаёт в промпт **факты** и следующий незаполненный вопрос анкеты —
  историю сообщений память не отдаёт вовсе.
* `save_context()` — отдельный LLM-вызов со structured output, потом слияние профиля и запись в файл:
  * пока идёт анкета — `SlotAnswer`: ответил ли пользователь на заданный вопрос и что ответил,
    плюс прочие факты (аллергии и т.п.); дополнительно из реплики берутся ещё не заданные
    «сущности» — имя, город вылета, направление, — чтобы потом о них не спрашивать;
  * после анкеты — `TravelerFacts`: пользователь может поменять любой факт («летим в Рим»).
* Защита от галлюцинаций извлечения: значение принимается, только если его слова есть в реплике
  пользователя (`grounded`); пустышки вроде «нет данных» и дубли (`covered`) отбрасываются.
  Всё отброшенное видно в служебной панели.

Факты извлекаются **до** генерации ответа: иначе агент не узнал бы, что пользователь
уже назвал имя, и переспросил бы его. Поэтому в `save_context` пара такая:
`inputs["input"]` — реплика пользователя, `outputs["output"]` — реплика ассистента, на которую он отвечал.

In [4]:
SLOT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ассистент задал пользователю вопрос анкеты про «{label}». Определи, ответил ли пользователь, и извлеки ответ.\n"
     "- «не знаю», «пока не решил», встречный вопрос без ответа — answered=false, value=null.\n"
     "- value — кратко, словами пользователя, а НЕ вариантами из вопроса, без потери деталей. "
     "В именительном падеже: \"Казань\", \"80 тысяч рублей\", \"середина ноября, 5 дней\".\n"
     "- other_facts — прочие важные факты о пользователе из реплики (не вопросы и не ответ на сам вопрос)."),
    ("human", "Вопрос: {question}\nРеплика пользователя: {answer}"),
])

PROFILE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты извлекаешь факты о пользователе-путешественнике из его ПОСЛЕДНЕЙ реплики.\n"
     "- Заполняй поле, только если пользователь сам сообщил или изменил это в реплике. Иначе null. Не выдумывай.\n"
     "- Значения пиши кратко, по-русски, в именительном падеже.\n"
     "- other_facts — только то, что не подходит ни к одному полю. Вопросы пользователя фактами не являются.\n\n"
     "Уже известно о пользователе:\n{known}"),
    ("human", "Реплика пользователя: {answer}"),
])

EMPTY_VALUES = {"", "-", "—", "нет", "нет данных", "неизвестно", "не указано", "null", "none"}
ENTITY_SLOTS = {"name", "home_city", "destination"}


def _stems(text: str) -> list[str]:
    """Грубые основы слов: «Казань»/«Казани» -> «казан», «жена»/«женой» -> «жен»."""
    words = re.findall(r"\w+", text.lower())
    return [w if w.isdigit() else w[:3] if len(w) <= 5 else w[:5] for w in words if len(w) >= 3 or w.isdigit()]


def grounded(value: str, text: str) -> bool:
    """Факт принимается, только если его слова есть в реплике пользователя.
    Маленькая модель иначе «переносит» в факты то, чего пользователь не говорил."""
    stems = _stems(value)
    return bool(stems) and any(stem in text.lower() for stem in stems)


def covered(fact: str, known: list[str]) -> bool:
    """Дубль: все основы слов факта уже есть в каком-то известном значении («ноябрь» ⊂ «середина ноября»)."""
    stems = _stems(fact)
    return any(all(stem in k.lower() for stem in stems) for k in known)


class SessionHistory(BaseChatMessageHistory):
    """История сообщений одного запуска — живёт только в оперативной памяти."""

    def __init__(self):
        self.messages: list[BaseMessage] = []

    def add_message(self, message: BaseMessage) -> None:
        self.messages.append(message)

    def clear(self) -> None:
        self.messages = []


class FactMemory(BaseChatMemory):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    store: FactStore
    user_id: str
    on_event: Callable[[str], None] | None = None
    profile: dict = Field(default_factory=dict)
    chat_memory: BaseChatMessageHistory = Field(default_factory=SessionHistory)  # историю здесь не храним
    slot_extractor: Any = None
    profile_extractor: Any = None

    def model_post_init(self, __context: Any) -> None:
        self.slot_extractor = SLOT_PROMPT | extract_llm.with_structured_output(SlotAnswer, method="json_schema")
        self.profile_extractor = PROFILE_PROMPT | extract_llm.with_structured_output(TravelerFacts, method="json_schema")
        self.profile = self.store.load(self.user_id)
        known = len(self.facts) + len(self.profile["other_facts"])
        source = f"{self.store.path} найден" if self.store.path.exists() else f"{self.store.path} нет — пустая память"
        self._emit(f"📂 {source}; профиль «{self.user_id}»: фактов {known}, "
                   f"осталось вопросов анкеты: {len(self.missing_slots())}")

    def _emit(self, message: str) -> None:
        if self.on_event:
            self.on_event(message)

    @property
    def facts(self) -> dict:
        return self.profile["facts"]

    def missing_slots(self) -> list[str]:
        return [slot for slot in PROFILE_QUESTIONS if not self.facts.get(slot)]

    def facts_text(self) -> str:
        lines = [f"- {FACT_LABELS[s]}: {v}" for s, v in self.facts.items() if v]
        lines += [f"- {fact}" for fact in self.profile["other_facts"]]
        return "\n".join(lines) or "(пока ничего)"

    @property
    def memory_variables(self) -> list[str]:
        return ["facts", "next_question"]

    def load_memory_variables(self, inputs: dict[str, Any]) -> dict[str, str]:
        missing = self.missing_slots()
        return {"facts": self.facts_text(), "next_question": PROFILE_QUESTIONS[missing[0]] if missing else ""}

    def save_context(self, inputs: dict[str, Any], outputs: dict[str, str]) -> None:
        answer, question = inputs["input"], outputs.get("output", "")
        missing = self.missing_slots()
        if missing:  # анкета: извлекаем ответ на заданный вопрос
            slot = missing[0]
            result: SlotAnswer = self.slot_extractor.invoke(
                {"label": FACT_LABELS[slot], "question": question, "answer": answer})
            if not result.answered:
                self._emit(f"⏭ на вопрос «{FACT_LABELS[slot]}» ответа нет — спросим ещё раз")
                candidates = {}
            else:
                value = result.value or ""
                if value.strip().lower() in EMPTY_VALUES or not grounded(value, answer):
                    self._emit(f"📝 {FACT_LABELS[slot]}: извлечено «{value}», в реплике этого нет — беру ответ как есть")
                    value = answer.strip().rstrip(".!")
                candidates = {slot: value}
            other_facts = result.other_facts
            # Пользователь мог сразу сказать и то, о чём ещё не спрашивали («я Мария из Новосибирска»).
            # От полного экстрактора берём только поля-сущности: их 3B-модель почти не путает.
            extra: TravelerFacts = self.profile_extractor.invoke({"known": self.facts_text(), "answer": answer})
            for other in ENTITY_SLOTS - {slot}:
                value = getattr(extra, other)
                if value and not self.facts.get(other) and not covered(value, list(candidates.values())):
                    candidates[other] = value
        else:  # свободный диалог: любой факт может появиться или измениться
            result: TravelerFacts = self.profile_extractor.invoke({"known": self.facts_text(), "answer": answer})
            candidates = result.model_dump(exclude={"other_facts"})
            other_facts = result.other_facts

        changes = {}
        for slot, value in candidates.items():
            value = (value or "").strip()
            if value.lower() in EMPTY_VALUES or covered(value, [self.facts.get(slot) or "\0"]):
                continue  # пусто или модель просто повторила уже известное
            if not grounded(value, answer):
                self._emit(f"🚫 отброшено, нет в реплике: {FACT_LABELS[slot]} = {value}")
                continue
            changes[slot] = (self.facts.get(slot), value)
            self.facts[slot] = value
        known = list(self.facts.values()) + self.profile["other_facts"]
        new_other = []
        for fact in other_facts:
            fact = fact.strip()
            if fact.lower() in EMPTY_VALUES or covered(fact, known):
                continue
            if not grounded(fact, answer):
                self._emit(f"🚫 отброшено, нет в реплике: доп. факт = {fact}")
                continue
            new_other.append(fact)
            known.append(fact)
        self.profile["other_facts"] += new_other
        # доп. факт мог стать дублем поля, заполненного позже («ноябрь» -> даты «середина ноября»)
        self.profile["other_facts"] = [f for f in self.profile["other_facts"]
                                       if not covered(f, list(self.facts.values()))]

        if not changes and not new_other:
            self._emit(f"🧠 «{answer[:40]}» → новых фактов нет")
            return
        for slot, (old, new) in changes.items():
            self._emit(f"🧠 {FACT_LABELS[slot]}: {new}" + (f"  (было: {old})" if old else ""))
        for fact in new_other:
            self._emit(f"🧠 доп. факт: {fact}")
        self.store.save(self.user_id, self.profile)
        self._emit(f"💾 записано в {self.store.path}")

    def clear(self) -> None:
        self.profile = {"facts": {}, "other_facts": []}
        self.store.delete(self.user_id)
        self._emit(f"🗑 профиль «{self.user_id}» удалён")

### Чат-цепочка

`RunnableWithMessageHistory` держит историю сообщений **только текущего запуска** (в оперативной памяти,
по `session_id`, класс `SessionHistory`); на диск попадают лишь факты. Следующий вопрос анкеты выбирает код
(первое пустое поле профиля), а LLM лишь коротко реагирует на реплику — так маленькая
модель задаёт вопросы строго по одному и не повторяет уже заданные.

In [5]:
CHAT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — TripMate, дружелюбный помощник по планированию путешествий. Отвечай по-русски, кратко: 1–3 предложения.\n\n"
     "Долговременная память — факты о пользователе:\n{facts}\n\n{mode}"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

MODE_INTERVIEW = ("Сейчас идёт анкета. Коротко и по-человечески отреагируй на реплику пользователя; "
                  "если он задал вопрос — ответь на него. НЕ задавай вопросов: следующий вопрос анкеты система добавит сама.")
MODE_FREE = ("Анкета заполнена. Помогай планировать поездку, опираясь на факты о пользователе. "
             "Не переспрашивай то, что уже знаешь.")
MODE_GREET_NEW = ("Это первый разговор. Одним предложением представься и скажи, что задашь несколько вопросов, "
                  "чтобы запомнить предпочтения. НЕ задавай вопросов сам.")
MODE_GREET_BACK = ("Пользователь вернулся после перерыва. Поприветствуй его по имени (если знаешь) и одним предложением "
                   "перечисли, что о нём помнишь. НЕ задавай вопросов сам.")

_sessions: dict[str, SessionHistory] = {}


def get_session_history(session_id: str) -> SessionHistory:
    return _sessions.setdefault(session_id, SessionHistory())


def drop_questions(text: str) -> str:
    """3B-модель всё равно норовит спросить что-то своё — во время анкеты такие предложения убираем."""
    sentences = re.split(r"(?<=[.!?…])\s+", text)
    return " ".join(s for s in sentences if not s.rstrip().endswith("?")).strip()


def _respond(inputs: dict) -> str:
    reply = (CHAT_PROMPT | chat_llm | StrOutputParser()).invoke(inputs).strip()
    question = inputs.get("next_question")
    if question or inputs["mode"] in (MODE_GREET_NEW, MODE_GREET_BACK):
        reply = drop_questions(reply)
    return "\n\n".join(part for part in (reply, question) if part)


chat_chain = RunnableWithMessageHistory(
    RunnableLambda(_respond),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

### Агент и служебная панель

`ServicePanel` выводит служебные события (загрузка файла, извлечённые факты, запись на диск)
и текущий профиль в **собственную ячейку**: она создаётся один раз и потом обновляется
на месте через `display_id`. Ячейки диалога показывают только реплики.

In [6]:
class ServicePanel:
    def __init__(self, title: str):
        self.title, self.events, self.memory = title, [], None
        self.handle = display(HTML(self._html()), display_id=True)

    def log(self, message: str) -> None:
        self.events.append(f"{datetime.now():%H:%M:%S}  {message}")
        self.refresh()

    def attach(self, memory: "FactMemory") -> None:
        self.memory = memory
        self.refresh()

    def refresh(self) -> None:
        self.handle.update(HTML(self._html()))

    def _html(self) -> str:
        style = "font:12px ui-monospace,monospace;background:#f6f8fa;color:#24292f;border:1px solid #d0d7de;border-radius:6px;padding:10px"
        events = "<br>".join(self.events[-25:]) or "—"
        facts = ""
        if self.memory:
            rows = "".join(f"<tr><td>{FACT_LABELS[s]}</td><td>{self.memory.facts.get(s) or '—'}</td></tr>"
                           for s in PROFILE_QUESTIONS)
            rows += "".join(f"<tr><td>доп.</td><td>{f}</td></tr>" for f in self.memory.profile["other_facts"])
            facts = f"<b>Профиль «{self.memory.user_id}»</b><table>{rows}</table>"
        return f"<div style='{style}'><b>🛠 {self.title}</b><br>{events}<hr>{facts}</div>"


class TripProfileAgent:
    EXIT = {"/exit", "выход", "exit", "quit"}

    def __init__(self, user_id: str, store: FactStore, panel: ServicePanel | None = None):
        self.user_id, self.panel = user_id, panel
        self.memory = FactMemory(store=store, user_id=user_id, on_event=panel.log if panel else None)
        if panel:
            panel.attach(self.memory)
        get_session_history(user_id).clear()  # новый запуск — новая история; факты остаются
        self.last_reply = ""

    def _reply(self, text: str, mode: str | None = None) -> str:
        variables = self.memory.load_memory_variables({})
        if mode is None:
            mode = MODE_INTERVIEW if variables["next_question"] else MODE_FREE
        self.last_reply = chat_chain.invoke(
            {"input": text, "mode": mode, **variables},
            config={"configurable": {"session_id": self.user_id}},
        )
        if self.panel:
            missing = self.memory.missing_slots()
            self.panel.log(f"❓ следующий вопрос анкеты: {missing[0]}" if missing else "✅ анкета заполнена")
        return self.last_reply

    def start(self) -> str:
        returning = bool(self.memory.facts or self.memory.profile["other_facts"])
        return self._reply("Привет!", MODE_GREET_BACK if returning else MODE_GREET_NEW)

    def send(self, text: str) -> str:
        self.memory.save_context({"input": text}, {"output": self.last_reply})
        return self._reply(text)

    def close(self) -> None:
        if self.panel:
            self.panel.log(f"👋 сессия завершена, факты лежат в {self.memory.store.path}")


def say(agent: TripProfileAgent, text: str) -> None:
    print(f"👤 {text}")
    print(f"🤖 {agent.send(text)}\n")


def repl(agent: TripProfileAgent) -> None:
    """Живой диалог через input(); выход — /exit."""
    print(f"🤖 {agent.start()}\n")
    while True:
        text = input("👤 ").strip()
        if text.lower() in agent.EXIT:
            agent.close()
            break
        if text:
            print(f"🤖 {agent.send(text)}\n")


store = FactStore(MEMORY_PATH)

## 2. Первый запуск

Профиль `alex` сбрасывается, чтобы демонстрация начиналась с пустой памяти.
Пользователь отвечает на часть анкеты, по пути отвлекается на свой вопрос, а затем выходит.

In [7]:
print("PID процесса ядра:", os.getpid())
store.delete("alex")

PID процесса ядра: 39113


**Служебная панель (сессия 1)** — обновляется по ходу диалога ниже:

In [8]:
panel = ServicePanel("Служебная информация — сессия 1")

Имя,Алексей
Город вылета,Казань
Куда хочет поехать,Стамбул
Планируемые даты поездки,"середина ноября, 5 дней"
Бюджет,—
С кем поедет,—
Интересы в путешествиях,—


In [9]:
agent = TripProfileAgent("alex", store, panel)
print(f"🤖 {agent.start()}\n")

🤖 Привет!

Как вас зовут?



In [10]:
say(agent, "Привет, я Алексей")
say(agent, "Живу в Казани, оттуда и летаю")
say(agent, "Хочу в Стамбул")

👤 Привет, я Алексей


🤖 Привет, Алексей!

Из какого города вы обычно вылетаете?

👤 Живу в Казани, оттуда и летаю


🤖 Отлично, Алексей! Вылет из Казани.

Куда хотите поехать?

👤 Хочу в Стамбул


🤖 Отлично, Алексей! Вы хотите посетить Стамбул.

Когда планируете поездку и на сколько дней?



Пользователь не отвечает на вопрос анкеты, а задаёт свой — агент отвечает и повторяет незакрытый вопрос:

In [11]:
say(agent, "Пока не знаю. А в ноябре там не холодно?")
say(agent, "Тогда в середине ноября, дней на пять")

👤 Пока не знаю. А в ноябре там не холодно?


🤖 Да, в ноябре в Стамбуле обычно тепло, около 10-15 градусов по Цельсию.

Когда планируете поездку и на сколько дней?

👤 Тогда в середине ноября, дней на пять


🤖 Отлично, Алексей! В середине ноября с тремя днями отпуска в Стамбуле у вас будет отличный отдых.

Какой бюджет на поездку?



In [12]:
agent.close()  # = команда выхода
print(MEMORY_PATH.read_text(encoding="utf-8"))

{
  "users": {
    "alex": {
      "facts": {
        "name": "Алексей",
        "home_city": "Казань",
        "destination": "Стамбул",
        "travel_dates": "середина ноября, 5 дней"
      },
      "other_facts": [],
      "updated_at": "2026-10-23T01:51:42"
    }
  }
}


## 3. После перезапуска

*Kernel → Restart*, затем снова выполнить раздел 1. PID другой — это новый процесс,
в оперативной памяти ничего не осталось. Агент читает `memory.json`, здоровается по имени
и продолжает анкету с первого незаполненного вопроса (бюджет).

In [7]:
print("PID процесса ядра:", os.getpid())

PID процесса ядра: 39368


**Служебная панель (сессия 2):**

In [8]:
panel = ServicePanel("Служебная информация — сессия 2 (после перезапуска)")

Имя,Алексей
Город вылета,Казань
Куда хочет поехать,Рим
Планируемые даты поездки,"середина ноября, 5 дней"
Бюджет,80 тысяч рублей
С кем поедет,с женой
Интересы в путешествиях,"Любим вкусно поесть и гулять по старым районам, музеи не очень"
доп.,аллергия на орехи


In [9]:
agent = TripProfileAgent("alex", store, panel)
print(f"🤖 {agent.start()}\n")

🤖 Алексей, планирует поездку в Стамбул с 5 дней в середине ноября.

Какой бюджет на поездку?



In [10]:
say(agent, "Тысяч 80 рублей на всё. Кстати, у меня аллергия на орехи")
say(agent, "С женой")
say(agent, "Любим вкусно поесть и гулять по старым районам, музеи не очень")

👤 Тысяч 80 рублей на всё. Кстати, у меня аллергия на орехи


🤖 Вот и отлично, Алексей. В Стамбуле есть много достопримечательностей, которые не содержат орехов.

С кем поедете — один, с партнёром, с детьми, с друзьями?

👤 С женой


🤖 Отлично, с женой. В Стамбуле можно посетить множество интересных мест, например, Собор Святого Саввы или мечеть Сулейман-челеби.

Что вы больше всего любите в путешествиях: музеи, пляж, еду, природу, прогулки?

👤 Любим вкусно поесть и гулять по старым районам, музеи не очень


🤖 Понял, Алексей. В Стамбуле вы сможете насладиться вкусной кухней и прогуляться по старым районам.



Анкета заполнена — дальше свободный диалог, в котором агент опирается на сохранённые факты:

In [11]:
say(agent, "Что ты обо мне знаешь?")
say(agent, "Посоветуй, чем заняться в первый день поездки")

👤 Что ты обо мне знаешь?


🤖 Я знаю, что Алексей планирует поездку в Стамбул с женой на 5 дней в середине ноября. Ваш бюджет составляет 80 тысяч рублей, аллергия на орехи, вы любите вкусно поесть и гулять по старым районам, музеи не очень интересуют. Город вылета — Казань.

👤 Посоветуй, чем заняться в первый день поездки


🤖 В первый день в Стамбуле можно прогуляться по старому городу, посетить площадь Тереканы и окрестности, а также попробовать местную кухню.



Изменение факта: новое значение перезаписывает старое в файле.

In [12]:
say(agent, "Слушай, мы передумали — летим в Рим")
agent.close()

👤 Слушай, мы передумали — летим в Рим


🤖 Конечно, Алексей. В Риме вы сможете посетить знаменитые музеи, прогуляться по древним улицам и окрестностям, а также насладиться вкусной итальянской кухней.



## 4. Несколько профилей

`user_id` (он же `session_id` для истории) разделяет пользователей: у нового профиля своя анкета,
профиль `alex` не затрагивается.

In [13]:
store.delete("maria")
panel_maria = ServicePanel("Служебная информация — профиль maria")

Имя,Мария
Город вылета,Новосибирск
Куда хочет поехать,—
Планируемые даты поездки,—
Бюджет,—
С кем поедет,—
Интересы в путешествиях,—


In [14]:
maria = TripProfileAgent("maria", store, panel_maria)
print(f"🤖 {maria.start()}\n")
say(maria, "Меня зовут Мария, я из Новосибирска")
maria.close()

🤖 Привет!

Как вас зовут?

👤 Меня зовут Мария, я из Новосибирска


🤖 Отлично, Мария!

Куда хотите поехать?



## 5. Итоговый файл памяти

In [15]:
print(MEMORY_PATH.read_text(encoding="utf-8"))

{
  "users": {
    "alex": {
      "facts": {
        "name": "Алексей",
        "home_city": "Казань",
        "destination": "Рим",
        "travel_dates": "середина ноября, 5 дней",
        "budget": "80 тысяч рублей",
        "companions": "с женой",
        "interests": "Любим вкусно поесть и гулять по старым районам, музеи не очень"
      },
      "other_facts": [
        "аллергия на орехи"
      ],
      "updated_at": "2026-10-23T01:52:07"
    },
    "maria": {
      "facts": {
        "name": "Мария",
        "home_city": "Новосибирск"
      },
      "other_facts": [],
      "updated_at": "2026-10-23T01:52:12"
    }
  }
}


## 6. Живой диалог

`INTERACTIVE = True`, чтобы поговорить с агентом через `input()`.
Выход — `/exit`; после перезапуска ядра и повторного запуска агент продолжит с того же места.

In [ ]:
INTERACTIVE = False
if INTERACTIVE:
    live_panel = ServicePanel("Служебная информация — живой диалог")
    repl(TripProfileAgent(input("user_id: ").strip() or "guest", store, live_panel))